# C1 — Kiểm chứng hai bước Newton tính tay bằng fit_irls (Kiên)
x = (−2, −1, 0, 1, 2), y = (0, 0, 1, 0, 1), Pr(y = 1 | x) = σ(w0 + w1·x), xuất phát (w0, w1) = (0, 0).

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from core import sigmoid
from logistic import fit_irls

x = np.array([-2, -1, 0, 1, 2], dtype=float)
y = np.array([0, 0, 1, 0, 1], dtype=float)

In [ ]:
# Các đại lượng trung gian của từng bước IRLS (để đối chiếu bảng tính tay)
Xt = np.c_[np.ones(5), x]            # cột 1 ứng với w0, cột 2 ứng với w1
theta = np.zeros(2)
for step in (1, 2):
    z = Xt @ theta
    p = sigmoid(z)
    s = p * (1 - p)
    u = z + (y - p) / s              # biến phản hồi làm việc
    g = Xt.T @ (p - y)               # gradient của loss TỔNG
    H = Xt.T @ (s[:, None] * Xt)     # Hessian của loss TỔNG
    rhs = Xt.T @ (s * u)
    print(f"--- Bước {step}, θ = {np.round(theta, 6)}")
    print(pd.DataFrame(dict(x=x, y=y, z=z, p=p, s=s, u=u)).round(4).to_string(index=False))
    print("g =", np.round(g, 4), "| H =", np.round(H, 4).tolist(), "| det H =", round(np.linalg.det(H), 4),
          "| X̃ᵀSu =", np.round(rhs, 4))
    theta = np.linalg.solve(H, rhs)
    print("θ mới =", np.round(theta, 6))

In [ ]:
# Đối chiếu với fit_irls (loss TRUNG BÌNH: bước Newton không đổi vì g và H cùng chia n)
w, b, h = fit_irls(x, y, lam=0.0, max_iter=2)
tab = pd.DataFrame(dict(k=range(len(h["b"])), w0=h["b"], w1=[wk[0] for wk in h["w"]],
                        loss_tb=h["loss"], grad_norm=h["grad_norm"]))
tab["loss_tong"] = 5 * tab.loss_tb
print(tab.round(6).to_string(index=False))

In [ ]:
# Nghiệm hợp lý cực đại (chạy tới hội tụ) để tham chiếu
w_star, b_star, h_star = fit_irls(x, y, lam=0.0, max_iter=100)
print("θ* = (%.6f, %.6f), số vòng = %d, loss tổng = %.6f"
      % (b_star, w_star[0], len(h_star["b"]) - 1, 5 * h_star["loss"][-1]))
tab.to_csv("../results/C1_newton.csv", index=False)